In [1]:
import os
os.chdir("../")
from src.PLSDA_BAYES import PLSDA
from src.rgb import img_rgb, imagen_prediccion
from src.preprocesado import snv, savgol
print(os.getcwd())
import pickle

from sklearn.model_selection import train_test_split

with open("DATA/arena_binned_sliced.pkl", "rb") as f:
    arena_plasticos = pickle.load(f)
arena_plasticosX=arena_plasticos.labelled_spectra
arena_plasticosY=arena_plasticos.ylabels

with open("DATA/gorliz_binned_sliced.pkl", "rb") as f:
    gorliz_plasticos = pickle.load(f)
gorliz_plasticosX=gorliz_plasticos.labelled_spectra
gorliz_plasticosY=gorliz_plasticos.ylabels
with open("DATA/sonabia_binned_sliced.pkl", "rb") as f:
    sonabia_plasticos = pickle.load(f)
sonabia_plasticosX=sonabia_plasticos.labelled_spectra
sonabia_plasticosY=sonabia_plasticos.ylabels
with open("DATA/sopela_binned_sliced.pkl", "rb") as f:
    sopela_plasticos = pickle.load(f)
sopela_plasticosX=sopela_plasticos.labelled_spectra
sopela_plasticosY=sopela_plasticos.ylabels

c:\Users\unaif\TFM PYTHON\Repositorio\TFM


In [2]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from chemotools.derivative import SavitzkyGolay
from chemotools.feature_selection import RangeCut
from chemotools.scatter import StandardNormalVariate
from sklearn.cross_decomposition import PLSRegression

param = {"plsda__n_components":range(1,21,4)}
model=PLSDA(max_iter=100)
pipeline = make_pipeline(
    
    SavitzkyGolay(window_length=5, polyorder=2,deriv=1), StandardNormalVariate(),
    StandardScaler(with_std=False),
    model)
modelo_ = GridSearchCV(pipeline,
                           param,
                           scoring='f1_weighted',
                           n_jobs=1,
                           refit=True,return_train_score=True,
                           cv=5)

In [8]:
import sklearn.metrics as met
import pandas as pd
import numpy as np
def metrica(
    X,
    y_true,
    modelo,
    playa_test,
    playa_train,
    modo,
    number_component
):
    y_true_clase = np.ravel(y_true)
    y_pred = np.ravel(modelo.predict(X))

    resultados = []

    resultados.append({
        "Playa_train": playa_train,
        "Playa_test": playa_test,
        "Preprocesado": modo,
        "PE_components": number_component,
        

        "Accuracy": met.accuracy_score(
            y_true_clase, y_pred
        ),

        "F1": met.f1_score(
            y_true_clase, y_pred,
            average="macro",
            zero_division=0
        ),

        "Precision": met.precision_score(
            y_true_clase, y_pred,
            average="macro",
            zero_division=0
        ),

        "Recall": met.recall_score(
            y_true_clase, y_pred,
            average="macro",
            zero_division=0
        )
    })

    return pd.DataFrame(resultados)


In [13]:
X_train=arena_plasticosX
y_train=arena_plasticosY

In [5]:
modelo=PLSDA()
modelo.fit(X_train, y_train)
print(modelo.thr)
prediccion=modelo.predict(X_train[0:1])

[0.46 0.36 0.38]


In [9]:

metrica(sonabia_plasticosX,
            sonabia_plasticosY,
            modelo,
            "Arena",
            "Sopela",
            "SNV",
            2)

,Playa_train,Playa_test,Preprocesado,PE_components,Accuracy,F1,Precision,Recall
0,Sopela,Arena,SNV,2,0.320037,0.342995,0.628314,0.323992


In [ ]:
def trainer(xtrain, y_train, modo, playa_trained):
    modelo=PLSDA()
    def optimize_SNV(model, xtrain, y_train_PE):
        pipeline = make_pipeline(
        StandardNormalVariate(),
        StandardScaler(with_std=False),
        model
    )
        modelo_ = GridSearchCV(
        pipeline,
        param,
        scoring="f1_weighted",
        n_jobs=1,
        cv=5)
        modelo_.fit(X_train, y_train)
        best_params_dict=modelo_.best_params_
        return best_params_dict

    def optimize_SAVGOL(model, xtrain, y_train_PE):
        pipeline = make_pipeline(
        SavitzkyGolay(window_length=5, polyorder=2, deriv=1),
        StandardNormalVariate(),
        StandardScaler(with_std=False),
        model)
        modelo_ = GridSearchCV(
        pipeline,
        param,
        scoring="f1_weighted",
        n_jobs=1,
        cv=5)
        modelo_.fit(X_train, y_train)
        best_params_dict=modelo_.best_params_
        return best_params_dict
    def optimize_SAVGOL_SNV(model, xtrain, y_train_PE):
        pipeline = make_pipeline(
        SavitzkyGolay(window_length=5, polyorder=2, deriv=1),
        StandardNormalVariate(),
        StandardScaler(with_std=False),
        model)
        modelo_ = GridSearchCV(
        pipeline,
        param,
        scoring="f1_weighted",
        n_jobs=1,
        cv=5
    )
        modelo_.fit(X_train, y_train)
        best_params_dict=modelo_.best_params_
        return best_params_dict
    match str(modo).lower():
        case "snv":
            best_params = optimize_SNV(
                modelo, xtrain, y_train)
            pipeline_multi = make_pipeline(
                StandardNormalVariate(),
                StandardScaler(with_std=False),
                modelo
            )

        case "savgol":
            best_params = optimize_SAVGOL(
                modelo, xtrain, y_train
            )

            pipeline_multi = make_pipeline(
                SavitzkyGolay(
                    window_length=5,
                    polyorder=2,
                    deriv=1
                ),
                StandardScaler(with_std=False),modelo            )

        case "savgolsnv":
            best_params = optimize_SAVGOL_SNV(
                modelo, xtrain, y_train)
            
            pipeline_multi = make_pipeline(
                StandardNormalVariate(),
                SavitzkyGolay(
                    window_length=5,
                    polyorder=2,
                    deriv=1
                ),
                StandardScaler(with_std=False),modelo            )
        case _:
            raise ValueError(f"Modo desconocido: {modo}")

    pipeline_multi.fit(xtrain, y_train)
    number_component = best_params
    resultados = pd.concat([
        metrica(
            arena_plasticosX,
            arena_plasticosY,
            pipeline_multi,
            "Arena",
            playa_trained,
            modo,
            number_component
        ),

        metrica(
            gorliz_plasticosX,
            gorliz_plasticosY,
            pipeline_multi,
            "Gorliz",
            playa_trained,
            modo,
            number_component
        ),

        metrica(
            sonabia_plasticosX,
            sonabia_plasticosY,
            pipeline_multi,
            "Sonabia",
            playa_trained,
            modo,
            number_component
        ),

        metrica(
            sopela_plasticosX,
            sopela_plasticosY,
            pipeline_multi,
            "Sopela",
            playa_trained,
            modo,
            number_component
        )
    ], ignore_index=True)

    return resultados

In [15]:
df1=trainer(arena_plasticosX,arena_plasticosY, snv, "Arena")
df1

AttributeError: 'function' object has no attribute 'lower'

In [53]:
df_final=pd.concat( [df1, df2,df3,df4,df5, df6,df7,df8, df9, df10,df11,df12],axis=0)
df_final

,cv,Arena_accuracy,Arena_f1,Arena_precision,Arena_recall,Gorliz_accuracy,Gorliz_f1,Gorliz_precision,Gorliz_recall,Sonabia_accuracy,Sonabia_f1,Sonabia_precision,Sonabia_recall,Sopela_accuracy,Sopela_f1,Sopela_precision,Sopela_recall
SNV Arena_PLS-DA,0.360973,0.762396,0.761136,0.777197,0.762396,0.749928,0.748786,0.769353,0.749928,0.622530,0.637987,0.704567,0.622530,0.702342,0.698277,0.727201,0.702342
SNV Gorliz,0.720628,0.900618,0.900414,0.913302,0.900618,0.978615,0.977768,0.978566,0.978615,0.817359,0.787500,0.892547,0.817359,0.879707,0.859065,0.901805,0.879707
SNV Sonabia,0.610934,0.552674,0.452542,0.459571,0.552674,0.513920,0.377810,0.754123,0.513920,0.744414,0.643522,0.576844,0.744414,0.566019,0.429203,0.521011,0.566019
SNV Sopela,0.842434,0.792666,0.792598,0.812804,0.792666,0.933629,0.933593,0.933721,0.933629,0.801293,0.808245,0.856874,0.801293,0.845160,0.845602,0.850694,0.845160
Sav-Gol Arena,0.908810,0.946837,0.947564,0.953164,0.946837,0.948175,0.953837,0.970164,0.948175,0.993167,0.993195,0.993353,0.993167,0.947488,0.950178,0.959412,0.947488
Sav-Gol Gorliz,0.896359,0.785397,0.747410,0.862151,0.785397,0.964262,0.958779,0.966603,0.964262,0.979501,0.979450,0.981031,0.979501,0.951634,0.948379,0.955420,0.951634
Sav-Gol Sonabia,0.989557,0.891071,0.886835,0.916340,0.891071,0.967537,0.969720,0.976221,0.967537,0.999631,0.999631,0.999631,0.999631,0.992607,0.992553,0.992737,0.992607
Sav-Gol Sopela,0.995133,0.978626,0.978665,0.979228,0.978626,0.966477,0.969134,0.977299,0.966477,1.000000,1.000000,1.000000,1.000000,0.997789,0.997785,0.997795,0.997789
Sav-Gol+SNV Arena,0.994048,0.993599,0.993598,0.993710,0.993599,0.966477,0.969134,0.977299,0.966477,1.000000,1.000000,1.000000,1.000000,0.997582,0.997576,0.997596,0.997582
Sav-Gol+SNV Gorliz,0.884199,0.988283,0.988277,0.988386,0.988283,0.973798,0.975510,0.980911,0.973798,1.000000,1.000000,1.000000,1.000000,0.996338,0.996325,0.996370,0.996338


In [54]:
df_final.to_csv("PLS-DA results.csv",";")

C:\Users\unaif\AppData\Local\Temp\ipykernel_2172\2203040132.py:1: FutureWarning: Starting with pandas version 3.0 all arguments of to_csv except for the argument 'path_or_buf' will be keyword-only.
  df_final.to_csv("PLS-DA results.csv",";")


In [ ]:
import pickle

with open("") as f:
    pickle.dump(model, f)

In [ ]:
import pickle

with open("") as f:
    pickle.dump(model, f)